# 00 - Setup do ambiente no Google Colab

Este notebook prepara o runtime do Colab para executar os experimentos do projeto:
1. Verifica a GPU (Tesla T4 ou superior),
2. Clona o repositório,
3. Instala as dependências (`requirements-colab.txt`),
4. Carrega o dataset ToLD-BR via Hugging Face (sem Google Drive),
5. Fornece uma célula para baixar resultados ao final da sessão,

## 1. Verificar GPU

In [1]:
!nvidia-smi

Sun Nov 15 22:54:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU: ", torch.cuda.get_device_name(0))
    print("Versão do PyTorch: ", torch.__version__)
    print("CUDA: ", torch.version.cuda)

CUDA disponível: True
GPU:  Tesla T4
Versão do PyTorch:  2.14.1+cu130
CUDA:  13.0


## 2. Clonar o Repósitório

In [3]:
import os

REPO_URL = "https://github.com/elianisaac/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais.git"
REPO_DIR = "Classifica-o-de-Discurso-T-xico-em-Redes-Sociais"

# Só clona se a pasta NÃO existir
if not os.path.exists(f"/content/{REPO_DIR}"):
    !git clone {REPO_URL} /content/{REPO_DIR}

# Força a entrada no caminho absoluto correto
%cd /content/{REPO_DIR}

Cloning into '/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 65 (delta 27), reused 54 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 481.05 KiB | 20.92 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/Classifica-o-de-Discurso-T-xico-em-Redes-Sociais


## 3. Instalar Dependencias

Usa `requirements-colab.txt`


In [ ]:
# Atualiza as ferramentas de build do Python
!pip install --upgrade pip setuptools wheel

# Força a instalação da bibliote datasets v3.0
!pip install datasets==3

# Instala as dependencias
!pip install -q -r requirements-colab.txt

  Attempting uninstall: datasets
    Found existing installation: datasets 2.21.0
    Uninstalling datasets-2.21.0:
      Successfully uninstalled datasets-2.21.0


In [16]:
# Verificação rápida das versões instaladas
import datasets, transformers, sklearn
print("datasets:", datasets.__version__)
print("transformers: ", transformers.__version__)
print("scikit-learn: ", sklearn.__version__)

datasets: 4.8.5
transformers:  5.22.1
scikit-learn:  1.6.1


# 4. Carregamento do Dataset

O script `src/load_data.py` baixa o dataset do Hugging Face (`JAugusto97/told-br`) e salva em disco (`data/processed/`) para evitar re-executar o *loading script* a cada uso.

In [17]:
!python src/load_data.py --variant binary --output data/processed/ --force

[2026-11-15 23:00:08] [INFO] [load_data] ============================================================
[2026-11-15 23:00:08] [INFO] [load_data] Processando variante: binary
[2026-11-15 23:00:08] [INFO] [load_data] ============================================================
[2026-11-15 23:00:08] [INFO] [load_data] Baixando do Hugging Face: JAugusto97/told-br (config=binary)
Generating train split: 100% 16800/16800 [00:00<00:00, 29929.80 examples/s]
Generating test split: 100% 2100/2100 [00:00<00:00, 31479.09 examples/s]
Generating validation split: 100% 2100/2100 [00:00<00:00, 31072.86 examples/s]
[2026-11-15 23:00:14] [INFO] [load_data] Salvando em disco: data/processed/told-br-binary
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 749046.55 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 384613.70 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 431027.08 examples/s]
[2026-11-15 23:00:14] [INFO] [load_data] Resumo do d

In [18]:
# Inspeção rápida do dataset carregado
from datasets import load_from_disk
ds = load_from_disk("data/processed/told-br-binary")
print(ds)
print("Exemplo: ", ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16800
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2100
    })
})
Exemplo:  {'text': 'rt @user olha quem chegouuuuu, nossos queridinhos!!! vem na minha direção 🤩\n\nfem 👧: 10/15\nmasc 👦: 25/40\n\n#qqmt https://t.co/jivoqeuq…', 'label': 0}


In [19]:
# Roda o pré-processamento gerando os textos limpos
!python src/preprocessing.py \
    --input data/processed/told-br-binary \
    --output data/processed/told-br-binary-clean

[2026-11-15 23:00:29] [INFO] [preprocessing] Carregando dataset de: data/processed/told-br-binary
[2026-11-15 23:00:30] [INFO] [preprocessing] Aplicando pré-processamento nas partições...
[2026-11-15 23:00:30] [INFO] [preprocessing]   Processando split: train
Map (num_proc=2): 100% 16800/16800 [00:01<00:00, 12770.03 examples/s]
[2026-11-15 23:00:31] [INFO] [preprocessing]   Processando split: test
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 8282.81 examples/s]
[2026-11-15 23:00:31] [INFO] [preprocessing]   Processando split: validation
Map (num_proc=2): 100% 2100/2100 [00:00<00:00, 8344.53 examples/s]
[2026-11-15 23:00:31] [INFO] [preprocessing] Salvando dataset processado em: data/processed/told-br-binary-clean
Saving the dataset (1/1 shards): 100% 16800/16800 [00:00<00:00, 1005211.30 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 441262.38 examples/s]
Saving the dataset (1/1 shards): 100% 2100/2100 [00:00<00:00, 450492.96 examples/s]
[2026-11-15 23:00:31

## 5. Executar os experimentos

Com o ambiente pronto e o dataset carregado, rode os scripts de treino. Exemplos:"

In [20]:
# Baseline: Regressão Logística
!python src/train_logistic.py --data data/processed/told-br-binary-clean --output results/models/

[2026-11-15 23:00:47] [INFO] [train_logistic] Carregando dados de: data/processed/told-br-binary-clean
[2026-11-15 23:00:47] [INFO] [train_logistic] Treino: 16800 amostras | Teste: 2100 amostras
[2026-11-15 23:00:47] [INFO] [train_logistic] Iniciando GridSearch (isso pode levar alguns minutos)...
Fitting 5 folds for each of 12 candidates, totalling 60 fits
[2026-11-15 23:01:22] [INFO] [train_logistic] Melhores parâmetros: {'clf__C': 1.0, 'clf__class_weight': 'balanced', 'clf__solver': 'liblinear'}
[2026-11-15 23:01:22] [INFO] [train_logistic] Melhor F1 (validação cruzada): 0.7070
[2026-11-15 23:01:22] [INFO] [train_logistic] 
=== Resultados no Teste ===
          f1: 0.6960
   precision: 0.6996
      recall: 0.6924
    accuracy: 0.7200
         auc: 0.8104
[2026-11-15 23:01:23] [INFO] [train_logistic] Modelo salvo em: results/models/logistic_regression_best.joblib
[2026-11-15 23:01:23] [INFO] [train_logistic] Concluído com sucesso.


In [21]:
# Rodando o BERTimbau
!python src/train_bertimbau.py \
    --data data/processed/told-br-binary-clean \
    --epochs 3 \
    --batch-size 16 \
    --output-dir results/models/bertimbau

[2026-11-15 23:02:01] [INFO] [train_bertimbau] Carregando dataset de: data/processed/told-br-binary-clean
[2026-11-15 23:02:01] [INFO] [train_bertimbau] Carregando tokenizador: neuralmind/bert-base-portuguese-cased
config.json: 100% 647/647 [00:00<00:00, 3.02MB/s]
tokenizer_config.json: 100% 43.0/43.0 [00:00<00:00, 190kB/s]
vocab.txt: 100% 210k/210k [00:00<00:00, 75.0MB/s]
added_tokens.json: 100% 2.00/2.00 [00:00<00:00, 8.18kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 518kB/s]
[2026-11-15 23:02:03] [INFO] [train_bertimbau] Tokenizando dados...
Map: 100% 16800/16800 [00:02<00:00, 8222.91 examples/s]
Map: 100% 2100/2100 [00:00<00:00, 7713.32 examples/s]
Map: 100% 2100/2100 [00:00<00:00, 6364.54 examples/s]
[2026-11-15 23:02:06] [INFO] [train_bertimbau] Carregando modelo...

pytorch_model.bin: downloading bytes:  53% 234M/438M [00:01<00:00, 250MB/s, 18.0MB/s  ]  
pytorch_model.bin: downloading bytes:  78% 341M/438M [00:02<00:00, 229MB/s, 28.6MB/s  ]
pytorch_model.bin: downloa

In [22]:
# Avaliação e comparação
!python src/evaluate.py --metrics-dir results/metrics --output-dir results/figures

[2026-11-15 23:28:23] [INFO] [evaluate] Buscando resultados em: results/metrics
[2026-11-15 23:28:23] [INFO] [evaluate] 
Tabela Comparativa:
| Modelo              |   F1-Score |   Precisão |   Recall |   IC 95% (Min) |   IC 95% (Max) |
|:--------------------|-----------:|-----------:|---------:|---------------:|---------------:|
| Regressão Logística |   0.695967 |   0.699584 | 0.692387 |       0.695967 |       0.695967 |
| BERTimbau           |   0.771003 |   0.722772 | 0.826132 |       0.771003 |       0.771003 |
[2026-11-15 23:28:23] [INFO] [evaluate] 
Gerando gráfico comparativo...
[2026-11-15 23:28:24] [INFO] [evaluate] Gráfico salvo em: results/figures/comparacao_modelos.png
